# Figure 5. Genetic regulation and colocalization signals at the ACE locus

In [ ]:
library(data.table)
library(tidyverse)
library(colocboost)
library(patchwork)
gene_id <- "ENSG00000159640"
gene_name <- "ACE"


## figure5a

In [ ]:
source("../Figure4/data/colocboost_plot.R")
Figure_5a <- readRDS("data/Figure_5a.rds")
start <- 4040
end <- 4234
ord <- c(10, 1, 3, 6)

# Unified robust-colocalization processing
Figure_5a_robust <- get_robust_colocalization(Figure_5a$cb_res, cos_npc_cutoff = 0.2, npc_outcome_cutoff = 0.2, pvalue_cutoff = 0.0001)
lead_var_5a <- Figure_5a_robust$cos_summary$top_variable[Figure_5a_robust$cos_summary$focal_outcome == "AD_Bellenguez"][1]
highlight_pos <- as.numeric(stringr::str_split(lead_var_5a, ":", simplify = TRUE)[, 2])

glyph <- function(count, pch, fill, border) {
  list(count = count, pch = pch, fill = fill, border = border)
}
glycan_annotation <- function(site, hexnac, hex, fuc = 0, neuac = 0) {
  parts <- list(
    glyph(hexnac, 22, "#2F7ED8", "#1F4E79"),
    glyph(hex, 21, "#68C83E", "#2E6B25")
  )
  if (fuc > 0) parts <- c(parts, list(glyph(fuc, 24, "#D84A4A", "#7F1D1D")))
  if (neuac > 0) parts <- c(parts, list(glyph(neuac, 23, "#9667CF", "#5B3A8A")))
  list(
    line1 = paste("ACE", site), glycan = parts,
    line_gap = 0.150, component_gap = 0.113, symbol_offset = 0.075
  )
}

outcome_annotations <- vector("list", length(Figure_5a$new_id_vector))
outcome_annotations[[1]] <- list(line1 = "ACE", line2 = "Protein", line_gap = 0.150)
outcome_annotations[[2]] <- glycan_annotation("N111", 2, 5)
outcome_annotations[[3]] <- glycan_annotation("N111", 6, 4, 1, 1)
outcome_annotations[[6]] <- glycan_annotation("N714", 2, 7)
outcome_annotations[[10]] <- list(line1 = "ADRD", line2 = "GWAS", line_gap = 0.150)

draw_gene_track <- function(x_range, tick_at, column, axis_size) {
  plot.new()
  plot.window(xlim = x_range, ylim = c(-0.35, 0.35), xaxs = "r", yaxs = "i")
  with(Figure_5a$intron_df, arrows(
    start, 0, end, 0, length = 0.045, angle = 25, code = 2, lwd = 0.7
  ))
  with(Figure_5a$ex_df, rect(
    start, -0.18, end, 0.18, col = "steelblue", border = NA
  ))
  axis(
    1, at = tick_at,
    labels = paste0(formatC(tick_at / 1e6, format = "f", digits = 2), "M"),
    cex.axis = 0.82 * axis_size
  )
  mtext("ACE", side = 1, line = 2.9, cex = 1.75, font = 2)
  box(lwd = 0.8)
}


In [ ]:
tmp <- tempfile(fileext = ".png")
draw_figure_5b <- function() {
  colocboost_plot(
    Figure_5a_robust,
    grange = start:end, variant_coord = TRUE, xlab = "",
    outcome_idx = ord, outcome_annotations = outcome_annotations,
    bottom_track = draw_gene_track, compact_x_axes = TRUE, track_height = 0.45,
    panel_top_margin = 0.2, outer_top_margin = 0,
    panel_left_margin = 6, panel_right_margin = 2.2,
    panel_inner_right_margin = 0.8, panel_column_widths = 1, plot_cols = 1,
    track_top_margin = 0.2,
    panel_axis_size = 2.0,
    panel_y_ticks = list(NULL, NULL, c(0, 1, 2, 3, 4), c(0, 4, 8, 12)),
    panel_y_limits = list(NULL, NULL, c(0, 5.3), NULL),
    add_highlight = TRUE, add_highlight_idx = highlight_pos,
    outcome_names = Figure_5a$new_id_vector, add_highlight_style = "coral_circle",
    outcome_legend_size = 1.5,
    annotation_scale = 1.45,
    lab_style = c(2.2, 1), axis_style = c(2.6, 1),
    genomic_ticks = c(63450000, 63500000),
    # # Mark variants belonging to a *different* CoS as "not colocalized" for outcomes not
    show_cos_to_uncoloc = TRUE
  )
}
png(tmp, width = 1578, height = 2600, res = 300)
draw_figure_5b()
dev.off()

img <- png::readPNG(tmp)
f5a_a <- ggplot() +
  annotation_raster(img, -Inf, Inf, -Inf, Inf) +
  theme_void()
f5a_a

### 5b

In [ ]:
f5b_img <- png::readPNG(paste0("data/", gene_name, "_Glycosites.png"))
options(repr.plot.width = 15, repr.plot.height = 6)
f5b_asset_aspect <- dim(f5b_img)[2] / dim(f5b_img)[1]
f5b_panel_aspect <- (17.1 * 4.9 / (4.9 + 4.6)) / (6.5 * 9 / 20)
f5b_asset_width <- min(1, f5b_asset_aspect / f5b_panel_aspect)
f5b_asset_height <- min(1, f5b_panel_aspect / f5b_asset_aspect)
f5b_asset_xmin <- (1 - f5b_asset_width) / 2
f5b_asset_xmax <- f5b_asset_xmin + f5b_asset_width
f5b_asset_ymin <- 1 - f5b_asset_height
f5b <- ggplot() +
  annotation_raster(f5b_img, f5b_asset_xmin, f5b_asset_xmax, f5b_asset_ymin, 1) +
  coord_cartesian(xlim = c(0, 1), ylim = c(0, 1), expand = FALSE) +
  theme_void()
f5b


### 5c

In [ ]:
figure_5c <- readRDS("data/Figure_5c.rds")
trait_levels <- c(
  "pQTL",
  "ACE\nHexNAc(6)Hex(4)Fuc(1)NeuAc(1)",
  "ACE\nHexNAc(2)Hex(7)"
)
figure_5c$plot_df <- figure_5c$plot_df %>%
  mutate(
    peptide_ID = factor(peptide_ID, levels = trait_levels),
    genotype = factor(genotype, levels = c("G/G", "G/A", "A/A"))
  )
med_df <- figure_5c$plot_df %>%
  group_by(genotype, peptide_ID) %>%
  summarise(median_y = median(pheno_value, na.rm = TRUE), .groups = "drop")


In [ ]:
make_panel_label_grob <- function(kind, site = NULL, counts = NULL) {
  is_glycan <- identical(kind, "glycan")
  children <- list(
    grid::textGrob(
      if (is_glycan) paste("ACE", site) else "ACE",
      x = grid::unit(0.97, "npc"), y = grid::unit(0.96, "npc"),
      just = c("right", "top"),
      gp = grid::gpar(fontsize = 11.5, fontface = "bold")
    ),
    grid::nullGrob()
  )
  if (!is_glycan) {
    children <- c(children, list(grid::textGrob(
      "Protein", x = grid::unit(0.97, "npc"), y = grid::unit(0.90, "npc"),
      just = c("right", "top"), gp = grid::gpar(fontsize = 11)
    )))
  } else {
    pch <- c(22, 21, 24, 23)
    fill <- c("#2F7ED8", "#68C83E", "#D84A4A", "#9667CF")
    border <- c("#1F4E79", "#2E6B25", "#7F1D1D", "#5B3A8A")
    symbol_x <- 0.94 - rev(seq_along(counts) - 1) * 0.135
    for (ii in seq_along(counts)) {
      children <- c(children, list(
        grid::textGrob(
          paste0(counts[ii], "×"),
          x = grid::unit(symbol_x[ii] - 0.018, "npc"),
          y = grid::unit(0.87, "npc"),
          just = c("right", "centre"), gp = grid::gpar(fontsize = 11.2)
        ),
        grid::pointsGrob(
          x = grid::unit(symbol_x[ii], "npc"), y = grid::unit(0.87, "npc"),
          pch = pch[ii], size = grid::unit(3.0, "mm"),
          gp = grid::gpar(fill = fill[ii], col = border[ii], lwd = 0.8)
        )
      ))
    }
  }
  do.call(grid::grobTree, children)
}

make_genotype_panel <- function(trait, label_grob) {
  panel_df <- figure_5c$plot_df %>% filter(peptide_ID == trait)
  panel_med <- med_df %>% filter(peptide_ID == trait)
  panel_range <- range(panel_df$pheno_value, na.rm = TRUE)
  panel_span <- diff(panel_range)
  if (!is.finite(panel_span) || panel_span <= 0) panel_span <- 1
  panel_limits <- c(
    panel_range[1] - 0.05 * panel_span,
    panel_range[2] + 0.25 * panel_span
  )
  panel_breaks <- scales::breaks_pretty(n = 4)(panel_range)
  panel_breaks <- panel_breaks[
    panel_breaks >= panel_limits[1] & panel_breaks <= panel_range[2]
  ]

  ggplot(panel_df, aes(x = genotype, y = pheno_value)) +
    geom_boxplot(width = 0.5, alpha = 0.7, outlier.shape = NA, color = "black") +
    geom_jitter(width = 0.2, size = 2, alpha = 0.6, aes(color = genotype)) +
    geom_line(
      data = panel_med,
      aes(x = as.numeric(genotype), y = median_y, group = 1),
      inherit.aes = FALSE, color = "grey50", linewidth = 1, linetype = "dashed"
    ) +
    scale_color_manual(values = figure_5c$geno_colors) +
    scale_y_continuous(limits = panel_limits, breaks = panel_breaks, expand = c(0, 0)) +
    annotation_custom(label_grob, xmin = -Inf, xmax = Inf, ymin = -Inf, ymax = Inf) +
    labs(
      x = "Genotype (rs8077276)",
      y = "Normalized abundance"
    ) +
    theme_bw(base_size = 13) +
    theme(
      axis.text.x = element_text(size = 13, face = "bold"),
      axis.text.y = element_text(size = 13),
      axis.title = element_text(size = 15, face = "bold"),
      axis.title.y = element_text(margin = margin(r = 5)),
      legend.position = "none",
      panel.grid = element_blank(),
      panel.border = element_rect(linewidth = 1.2),
      plot.margin = margin(t = 4, r = 9, b = 4, l = 9)
    )
}

f5c_protein <- make_genotype_panel(
  trait_levels[1], make_panel_label_grob("protein")
)
f5c_complex <- make_genotype_panel(
  trait_levels[2], make_panel_label_grob("glycan", "N111", c(6, 4, 1, 1))
)
f5c_man7 <- make_genotype_panel(
  trait_levels[3], make_panel_label_grob("glycan", "N714", c(2, 7))
)
f5c <- patchwork::wrap_plots(f5c_protein, f5c_complex, f5c_man7, nrow = 1) +
  patchwork::plot_layout(axis_titles = "collect")


## Combine

In [ ]:
library(ggplot2)
library(cowplot)
library(magick)
library(patchwork)

left_panel_width <- 17.1 * 4.9 / (4.9 + 4.6)
right_panel_width <- (17.1 - left_panel_width) / 2
panel_overlap <- 0.25
figure_width <- left_panel_width + right_panel_width - panel_overlap
figure_height <- 6.5
options(repr.plot.width = figure_width, repr.plot.height = figure_height)

f5b_aligned <- cowplot::ggdraw() +
  cowplot::draw_plot(f5a_a, x = 0.031, y = 0.020, width = 0.914, height = 0.975)
f5c_aligned <- cowplot::ggdraw() +
  cowplot::draw_plot(f5c, x = 0, y = 0, width = 1, height = 1) +
  cowplot::draw_label(
    "c", x = 0, y = 0.995, hjust = 0, vjust = 1,
    fontface = "bold", size = 22
  )

a_panel <- f5b + labs(tag = "a") +
  theme(
    plot.tag = element_text(size = 22, face = "bold"),
    plot.tag.position = c(0.02, 0.98)
  )
left_column <- a_panel + f5c_aligned +
  plot_layout(ncol = 1, heights = c(9, 11))
right_column <- f5b_aligned + labs(tag = "b") +
  theme(
    plot.tag = element_text(size = 22, face = "bold"),
    plot.tag.position = c(0.005, 0.98)
  )

combined_fig <- cowplot::ggdraw() +
  cowplot::draw_plot(
    left_column, x = 0, y = 0,
    width = left_panel_width / figure_width, height = 1
  ) +
  cowplot::draw_plot(
    right_column, x = (left_panel_width - panel_overlap) / figure_width, y = 0,
    width = right_panel_width / figure_width, height = 1
  )

combined_fig

In [ ]:
figure5_uncropped <- tempfile(fileext = ".pdf")
figure5_cropped <- tempfile(fileext = ".pdf")

ggsave(
  filename = figure5_uncropped, plot = combined_fig,
  device = grDevices::cairo_pdf,
  width = figure_width, height = figure_height, units = "in", dpi = 300,
  limitsize = FALSE, bg = "white"
)

crop_status <- system2(
  "pdfcrop",
  c("--margins", shQuote("5 10 5 10"), figure5_uncropped, figure5_cropped)
)
if (!identical(crop_status, 0L)) stop("pdfcrop failed")
file.copy(figure5_cropped, "Figure5.pdf", overwrite = TRUE)
